In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week7-lesson-5"). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

### caching external table

In [2]:
spark.sql("use itv024128")

""


In [3]:
spark.sql(" drop table itv024128.orders1GB_ext ")

""


In [4]:
spark.sql(" create table itv024128.orders1GB_ext ( order_id long, order_date string , cust_id long, status string) using csv location '/user/itv024128/orders_wh' ")

""


In [5]:
spark.sql("cache table itv024128.orders1GB_ext") 

""


In [6]:
spark.sql("select count(*) from itv024128.orders1GB_ext")

count(1)
137770


In [7]:
spark.sql("insert into itv024128.orders1GB_ext values(  11111, CAST('2013-07-25' AS DATE),  11599 , 'BOOKED')")

""


#### an insert to cached table will invalidate the cache automatically

#### any subsequent queries after an insert to a cached table will reload the cache automatically

In [8]:
spark.sql("select count(*) from itv024128.orders1GB_ext")

count(1)
137771


In [9]:
spark.sql("select count(*) from itv024128.orders1GB_ext")

count(1)
137771


#### Add a new file to external table location hadoop fs -cp /user/itv024128/orders_wh/orders_wh.csv /user/itv024128/orders_wh/orders_wh1.csv, But the queries from cached table will show old data

In [10]:
spark.sql("select count(*) from itv024128.orders1GB_ext")

count(1)
137771


### stale cache

#### spark will still get data from cache. New data is not picked up as new data was loaded manually by adding a file and not using an insert operation

### command to refresh table with new file in location

In [12]:
spark.catalog.refreshTable("itv024128.orders1GB_ext")

In [13]:
# oR use spark.ssql("refresh table itv024128.orders1GB_ext") OR ## uncache and cache the table again

In [14]:
spark.sql("select count(*) from itv024128.orders1GB_ext")

count(1)
137771


### Manually removed a file from the table's external location

In [19]:
spark.sql("select count(*) from itv024128.orders1GB_ext")

count(1)
137771


### Count is still showing old value

In [22]:
spark.sql("refresh table itv024128.orders1GB_ext")

""


## After refresh the count is now showing correct value

In [23]:
spark.sql("select count(*) from itv024128.orders1GB_ext")

count(1)
68887
